# SharePoint Verification
Confirm SharePoint connection has been configured and docs can be ingested and parsed. 

---


## Unity Catalog Connection
Create a Unity Catalog connection to store your SharePoint credentials. 
https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/sharepoint-connection
- **OAuth U2M: Databricks-managed** - The account used to authenticate is shared with all users.  Use a service account!
- **OAuth U2M: Custom-managed** - organization requires control over the OAuth application
- **OAuth M2M** - Use M2M for fully automated production pipelines that run without user interaction.

Create the connection manually through the Catalog Explorer UI, then update the connection_name in the Variable definitions cell, below.

In [0]:
dbutils.widgets.text("catalog_name", "dennis_schultz_azure_classic01", "Catalog Name")
dbutils.widgets.text("schema_name", "sharepoint_testing", "Schema Name")
dbutils.widgets.text("volume_name", "image_storage", "Volume Name")

# Unity Catalog
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
bronze_table_name = "01_bronze_documents_raw"
silver_table_name = "02_silver_documents_parsed"
image_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/images"

# SharePoint
connection_name = "vantage_demo_sharepoint_connection"
sharepoint_site_url = "https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/Forms/AllItems.aspx"

excel_sharepoint_file_url = "https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/Sample-Superstore.xls"
excel_bronze_table_name = "01_bronze_documents_excel"
excel_dataAddress = "Orders"

In [0]:
spark.sql(f"""
  CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
""")
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}
""")

In [0]:
# Set the default catalog and schema for this session
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [0]:
if spark.catalog.tableExists(bronze_table_name):
    spark.sql(f"DELETE FROM {bronze_table_name}")

if spark.catalog.tableExists(silver_table_name):
    spark.sql(f"DELETE FROM {silver_table_name}")

## Unstructured

### PDF documents

In [0]:
# Read all PDF files from SharePoint as binary files
# This is a batch read. For automatic incremental ingestion, view the cells at the bottom of the notebook to see how to use this connector in Lakeflow Spark Declarative Pipelines
pdf_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", connection_name) # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")  # Only ingest PDF files
    .load(sharepoint_site_url)
    .select("*", "_metadata")
)

# Display the DataFrame to see the PDF files
display(pdf_df.limit(1))

# Save the PDF files to a Delta table for persistent storage
pdf_df.write \
    .mode("append") \
    .saveAsTable(bronze_table_name)


### PowerPoint documents

In [0]:
# Read all Powerpoint files from SharePoint as binary files

ppt_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", connection_name) # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pptx")  # Only ingest PowerPoint files
    .load(sharepoint_site_url)
    .select("*", "_metadata")
)

# Save the Powerpoint files to a Delta table for persistent storage
ppt_df.write \
    .mode("append") \
    .saveAsTable(bronze_table_name)

# Display the DataFrame with only the first 100 bytes of file content
display(
    ppt_df.selectExpr(
        "path",
        "modificationTime",
        "length",
        "substring(content, 1, 100) as content_first_100_bytes",
        "_metadata"
    )
)

## Parse all docs using `ai_parse_document()`

When ingesting unstructured files from SharePoint (such as PDFs, Word documents, or PowerPoint files) using the standard SharePoint connector with binaryFile format, the file contents are stored as raw binary data.

To prepare these files for AI workloads—such as RAG, search, classification, or document understanding—you can parse the binary content into structured, queryable output by applying `ai_parse_document()` onto the `content` column.

In [0]:

#  Parse ALL documents in the Delta table and extract text using AI
parsed_df = (spark.sql(f"""
  SELECT
    *,
    ai_parse_document(
        content,
        MAP(
          'version', '2.0', 
          'imageOutputPath', '{image_path}'
        )
    ) AS parsed_content
  FROM
    {bronze_table_name}
  LIMIT 5 -- limiting to only the first 5 PDFs
  """))

parsed_df.write \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)


display(parsed_df)

You can also use `ai_parse_document()` **within Lakeflow Spark Declarative Pipelines to enable incremental parsing**. As new files stream in from SharePoint, they are automatically parsed as your pipeline updates.

## Structured


### Sync Individual Excel Files to Delta Tables

This section demonstrates how to:
1. Read a specific Excel file from SharePoint using Python `spark.read()` or SQL `read_files()`

Excel files support options like `headerRows` and `dataAddress` to specify which sheet and range to read.

In [0]:
# Read a specific Excel file from SharePoint
excel_df = (spark.read
    .format("excel")
    .option("databricks.connection", connection_name)
    .option("headerRows", 1)  # First row contains headers
    .option("inferSchema", True)  # Automatically infer column types
    .option("dataAddress", excel_dataAddress) # Specify the tab and cell range, if required
    .load(excel_sharepoint_file_url)
)

# Save the Excel data to a Delta table with column mapping enabled (enables the use of non-standard column names)
excel_df.write \
    .option("delta.columnMapping.mode", "name") \
    .mode("overwrite") \
    .saveAsTable(excel_bronze_table_name)

# Display the Excel data
display(excel_df)